In [52]:
# ── Step 1: RHF + CCSD on H₂ ──────────────────────────────────────────────
# PySCF gives us: MO coefficients, one-body integrals, t1/t2 amplitudes,
# and FCI as a reference.

import numpy as np
from pyscf import gto, scf, cc

# mol = gto.M(atom="H 0 0 0; H 0 0 2", basis="sto6g", unit='B', verbose=0)
a = 2 # bond length in a cluster
d = 100 # distance between each cluster
unit = 'b' # unit of length
na = 2 # size of a cluster (monomer)
nc = 2 # set as integer multiple of monomers
spin = 0 # spin per monomer
frozen = 0 # frozen orbital per monomer
elmt = 'H'
basis = 'sto6g'
# for nc in nc_list:
atoms = ""
for n in range(nc * na):
    shift = ((n - n % na) // na) * (d-a)
    atoms += f"{elmt} {n*a+shift:.5f} 0.00000 0.00000 \n"
mol = gto.M(atom=atoms,basis=basis,spin=spin*nc,unit=unit,verbose=4)

mf = scf.RHF(mol)
mf.kernel()
print(f"RHF  energy: {mf.e_tot:.10f} Ha")

mycc = cc.CCSD(mf)
mycc.kernel()
print(f"CCSD energy: {mycc.e_tot:.10f} Ha")

System: uname_result(system='Linux', node='sharmagroup-rn', release='6.17.0-19-generic', version='#19~24.04.2-Ubuntu SMP PREEMPT_DYNAMIC Fri Mar  6 23:08:46 UTC 2', machine='x86_64')  Threads 16
Python 3.11.14 (main, Oct 21 2025, 18:31:21) [GCC 11.2.0]
numpy 2.3.1  scipy 1.16.2  h5py 3.14.0
Date: Mon Mar 23 17:04:31 2026
PySCF version 2.12.1
PySCF path  /home/sharmagroup/sharmagroup/pyscf
GIT ORIG_HEAD 3d1768f5e33b144b606c3d2c81c12ee54d794501
GIT HEAD (branch master) f0861da51f017364d8bbaa20b742a94f3733305f

[ENV] PYSCF_EXT_PATH /home/sharmagroup/sharmagroup/pyscf-forge
[CONFIG] conf_file None
[INPUT] verbose = 4
[INPUT] num. atoms = 4
[INPUT] num. electrons = 4
[INPUT] charge = 0
[INPUT] spin (= nelec alpha-beta = 2S) = 0
[INPUT] symmetry False subgroup None
[INPUT] Mole.unit = b
[INPUT] Symbol           X                Y                Z      unit          X                Y                Z       unit  Magmom
[INPUT]  1 H      0.000000000000   0.000000000000   0.000000000000 AA    

In [53]:
# ── Step 2: configure JAX and define orbital dimensions ───────────────────

# import sys, os
# sys.path.insert(0, os.path.join("..", "ad_afqmc_prototype"))

from ad_afqmc_prototype import config
config.configure_once()   # enable float64, etc.

import jax
import jax.numpy as jnp

mo   = mf.mo_coeff          # (nao, norb)  MO coefficient matrix
norb = mo.shape[1]           # number of MOs
nocc = mol.nelectron // 2   # occupied MOs (closed-shell)
nvir = norb - nocc
print(f"norb={norb}  nocc={nocc}  nvir={nvir}")

norb=4  nocc=2  nvir=2


In [123]:
from ad_afqmc_prototype.staging import TrialInput

def _stage_pt2ccsd_input(obj):
    if obj.kind != "ccsd":
        raise ValueError(f"Unreachable: '{obj.kind}'.")
    
    t1 = obj.t1
    t2 = obj.t2

    t1 = np.asarray(t1)
    t2 = np.asarray(t2)
    t2 = t2.transpose(0, 2, 1, 3) # (i,j,a,b) -> (i,a,j,b)

    def _thouless(init_slater, t1):
        # Thouless transformation: |psi'> = exp(t1_ia a+ i)|psi>
        # init slater: mo_coeff of psi (in mo basis)
        # return mo_coeff of psi' (in mo basis)
        norb, nvir = t1.shape
        norb = nocc + nvir
        exp_t1 = np.eye(norb, dtype=np.float64)
        exp_t1[:nocc, nocc:] = t1
        # exp_t1 = jsp.linalg.expm(t1_full)
        return exp_t1.T @ init_slater
    
    norb, nvir = t1.shape
    norb = nocc + nvir
    mo_coeff = np.eye(norb, dtype=jnp.float64)[:,:nocc]
    mo_t = _thouless(mo_coeff, t1)

    data = {"mo": mo_coeff, "mo_t": mo_t, "t2": t2}
    return TrialInput(
        kind="pt2ccsd",
        data=data,
        norb_frozen=obj.norb_frozen,
        source_kind=obj.source,
    )

from dataclasses import dataclass
from jax import tree_util

@tree_util.register_pytree_node_class
@dataclass(frozen=True)
class Pt2ccsdTrial:
    """
    Restricted pt2CCSD trial in an MO basis where the reference
    determinant occupies the first nocc orbitals.

    Arrays:
      mo_coeff: (nocc, norb)               mo_coeff of |psi_0> in mo basis
      mo_t: (nocc, norb)                   mo_coeff |psi'> = e^t1 |psi_0> by Thouless Theorem in mo basis
      t2: (nocc, nvir, nocc, nvir)         doubles amplitudess t2_{i a j b} in mo basis
    """

    # t1: jax.Array
    # mo_coeff: jax.Array  # (norb, nocc)
    mo_t: jax.Array # (norb, nocc)
    t2: jax.Array # (norb, nvir, norb, nvir)

    @property
    def nocc(self) -> int:
        return int(self.t2.shape[0])

    @property
    def nvir(self) -> int:
        return int(self.t2.shape[1])

    @property
    def norb(self) -> int:
        return int(self.nocc + self.nvir)

    def tree_flatten(self):
        # children = (self.mo_coeff, self.mo_t, self.t2)
        children = (self.mo_t, self.t2)
        aux = None
        return children, aux

    @classmethod
    def tree_unflatten(cls, aux, children):
        # mo_coeff, mo_t, t2 = children
        mo_t, t2 = children
        return cls(
            # mo_coeff = mo_coeff,
            mo_t = mo_t,
            t2 = t2,
        )

In [124]:
# my step 3 & 4 build the Hamiltonian and the trial
# just use what is availble in the code, don't do unnecessary tensor contractions

from ad_afqmc_prototype.staging import _stage_ham_input, _stage_trial_input
from ad_afqmc_prototype.staging import StagedMfOrCc
from ad_afqmc_prototype.ham.chol import HamChol
from ad_afqmc_prototype.trial.rhf import RhfTrial
from ad_afqmc_prototype.trial.cisd import CisdTrial

objmf = StagedMfOrCc(mycc._scf, norb_frozen=None)
staged_ham = _stage_ham_input(objmf, chol_cut=1e-5, verbose=False)
staged_guide = _stage_trial_input(objmf)

ham_data = HamChol(
    h0=jnp.array(staged_ham.h0),
    h1=jnp.array(staged_ham.h1),
    chol=jnp.array(staged_ham.chol),
    basis=staged_ham.basis,
)

guide_data = RhfTrial(
    mo_coeff=jnp.array(staged_guide.data['mo'][:,:nocc]),
    )

objcc = StagedMfOrCc(mycc, norb_frozen=None)
# staged_trial = _stage_trial_input(objcc)
# trial_data = CisdTrial(
#     ci1 = staged_trial.data["ci1"],
#     ci2 = staged_trial.data["ci2"],
#     )
staged_trial = _stage_pt2ccsd_input(objcc)

trial_data = Pt2ccsdTrial(
    mo_t = staged_trial.data["mo_t"],
    t2 = staged_trial.data["t2"],
    )

In [128]:
@dataclass(frozen=True)
class Pt2ccsdMeasCfg:
    memory_mode: str = "low"  # or Literal["low","high"]
    mixed_real_dtype: jnp.dtype = jnp.float64
    mixed_complex_dtype: jnp.dtype = jnp.complex128
    mixed_real_dtype_testing: jnp.dtype = jnp.float32
    mixed_complex_dtype_testing: jnp.dtype = jnp.complex64

@tree_util.register_pytree_node_class
@dataclass(frozen=True)
class Pt2ccsdMeasCtx:
    # half-rotated:
    # rot_h1: jax.Array  # (nocc, norb)
    # rot_chol: jax.Array  # (n_chol, nocc, norb)
    # rot_chol_flat: jax.Array  # (n_chol, nocc*norb)
    cfg: Pt2ccsdMeasCfg  # static

    def tree_flatten(self):
        children = ()
        aux = (self.cfg,)
        return children, aux

    @classmethod
    def tree_unflatten(cls, aux, children):
        (cfg,) = aux
        # rot_chol = children
        return cls(cfg=cfg)

def build_meas_ctx(
    ham_data: HamChol, trial_data: Pt2ccsdTrial, cfg: Pt2ccsdMeasCfg = Pt2ccsdMeasCfg()
) -> Pt2ccsdMeasCtx:
    if ham_data.basis != "restricted":
        raise ValueError("pt2CCSD MeasOps currently assumes HamChol.basis == 'restricted'.")

    # chol = ham_data.chol  # (n_chol, norb, norb)
    # nocc = trial_data.nocc
    # rot_chol = chol[:, :nocc, :]  # (n_chol, nocc, norb)
    # cH = trial_data.mo_coeff.conj().T  # (nocc, norb)
    # rot_h1 = cH @ ham_data.h1  # (nocc, norb)
    # rot_chol = jnp.einsum("pi,gij->gpj", cH, ham_data.chol, optimize="optimal")
    # rot_chol_flat = rot_chol.reshape(rot_chol.shape[0], -1)

    return Pt2ccsdMeasCtx(cfg=cfg)

In [ ]:
# staged_trial = _stage_pt2ccsd_input(obj)

# trial_data = Pt2ccsdTrial(
#     mo_coeff=jnp.array(staged_trial.data['mo'][:,:nocc]),
#     mo_t = staged_trial.data["mo_t"],
#     t2 = staged_trial.data["t2"],
#     )

In [130]:
# ── Step 5: build Pt2CCSDMeasCtx — precomputed measurement arrays ────────────
#
# Two arrays cached once to avoid recomputation inside the energy kernel:
#   rot_chol[g, p, q] = L^MO_{g, p<nocc, q}  (occupied-row slice of Cholesky)
#   lci1[g, p, i]     = sum_t L^MO_{g,pt} * ci1_{it}  (Cholesky × singles amplitudes)

# from ad_afqmc_prototype.meas.cisd import build_meas_ctx, CisdMeasCfg

trial_cfg      = Pt2ccsdMeasCfg(memory_mode="low")  # vectorise over all Cholesky at once
trial_meas_ctx = build_meas_ctx(ham_data, trial_data, trial_cfg)

# print(f"rot_chol shape: {trial_meas_ctx.rot_chol.shape}   (nchol, nocc, norb)")
print(trial_meas_ctx)

Pt2ccsdMeasCtx(cfg=Pt2ccsdMeasCfg(memory_mode='low', mixed_real_dtype=<class 'jax.numpy.float64'>, mixed_complex_dtype=<class 'jax.numpy.complex128'>, mixed_real_dtype_testing=<class 'jax.numpy.float32'>, mixed_complex_dtype_testing=<class 'jax.numpy.complex64'>))


In [ ]:
# ── Step 6: import the energy and overlap kernels ─────────────────────────

# from ad_afqmc_prototype.meas.cisd import energy_kernel_rw_rh
# from ad_afqmc_prototype.trial.cisd import overlap_r

In [131]:
from jax import lax
# from ad_afqmc_prototype.trial.rhf import overlap_r
def overlap_r(walker: jax.Array, trial_data: Pt2ccsdTrial) -> jax.Array:
    # <exp(T1)HF|walker>
    return jnp.linalg.det(trial_data.mo_t.T.conj() @ walker)**2

# @jax.jit
def _greens_restricted(walker: jax.Array, mo_t: jax.Array) -> jax.Array:
    return (walker @ (jnp.linalg.inv(mo_t.T @ walker)) @ mo_t.T).T

# @jax.jit
def _greenp_from_green(green: jax.Array, nocc: int) -> jax.Array:
    norb = green.shape[0]
    return (green - jnp.eye(norb))[:,nocc:]

# @jax.jit
def energy_kernel_rw_rh(
    walker: jax.Array, ham_data: HamChol, meas_ctx: Pt2ccsdMeasCtx, trial_data: Pt2ccsdTrial
) -> jax.Array:
    mo_t, t2 = trial_data.mo_t, trial_data.t2
    nocc = trial_data.nocc

    green = _greens_restricted(walker, mo_t)  # (norb, norb)
    greenp = _greenp_from_green(green, nocc)  # (norb, nvir)

    # h0 = ham_data.h0
    h1 = ham_data.h1
    chol = ham_data.chol
    # rot_chol = meas_ctx.rot_chol

    hg = jnp.einsum("pq,pq->", h1, green, optimize="optimal")
    e1_0 = 2 * hg

    # double excitations
    t2g_c = jnp.einsum("iajb,ia->jb", t2, green[:nocc,nocc:], optimize="optimal")
    t2g_e = jnp.einsum("iajb,ib->ja", t2, green[:nocc,nocc:], optimize="optimal")
    t2_green_c = (greenp @ t2g_c.T) @ green[:nocc,:]
    t2_green_e = (greenp @ t2g_e.T) @ green[:nocc,:]
    t2_green = 2 * t2_green_c - t2_green_e
    t2g = 2 * t2g_c - t2g_e
    gt2g = jnp.einsum("ia,ia->", t2g, green[:nocc,nocc:], optimize="optimal")
    e1_2_1 = 2 * hg * gt2g
    e1_2_2 = -2 * jnp.einsum("pq,pq->", h1, t2_green, optimize="optimal")
    e1_2 = e1_2_1 + e1_2_2 # <exp(T1)HF|T2 h1|walker>/<exp(T1)HF|walker>

    # two body energy
    lg = jnp.einsum("gpq,pq->g", chol, green, optimize="optimal")

    # double excitations
    lt2g = jnp.einsum("gpq,pq->g", chol, t2_green, optimize="optimal")
    e2_2_2_1 = -lt2g @ lg

    def scanned_fun(carry, x):
        chol_i = x
        # e2_0
        gl_i = jnp.einsum("pr,qr->pq", green, chol_i, optimize="optimal")
        e2_0_1_i = (2*jnp.trace(gl_i))**2 / 2.0
        e2_0_2_i = - jnp.einsum('pq,qp->',gl_i,gl_i, optimize="optimal")
        carry[0] += e2_0_1_i + e2_0_2_i
        # e2_2_2_2
        lt2_green_i = jnp.einsum("pr,qr->pq", chol_i, t2_green, optimize="optimal")
        carry[1] += 0.5 * jnp.einsum("pq,pq->", gl_i, lt2_green_i, optimize="optimal")
        # e2_2_3
        glgp_i = jnp.einsum("iq,qa->ia", gl_i[:nocc,:], greenp, optimize="optimal")
        l2t2_1 = jnp.einsum("ia,jb,iajb->", glgp_i, glgp_i, t2, optimize="optimal")
        l2t2_2 = jnp.einsum("ib,ja,iajb->", glgp_i, glgp_i, t2, optimize="optimal")
        carry[2] += 2 * l2t2_1 - l2t2_2
        return carry, 0.0

    [e2_0, e2_2_2_2, e2_2_3], _ = lax.scan(scanned_fun, [0.0,0.0,0.0], chol)
    e2_2_1 = e2_0 * gt2g
    e2_2_2 = 4 * (e2_2_2_1 + e2_2_2_2)
    e2_2 = e2_2_1 + e2_2_2 + e2_2_3

    # o0 = jnp.linalg.det(walker[:nocc,:nocc]) ** 2
    # t1 = jnp.linalg.det(mo_t.T.conj() @ walker)**2 / o0 # <exp(T1)HF|walker>/<HF|walker>
    t2 = gt2g # * t1 # <exp(T1)HF|T2|walker>/<HF|walker>
    e0 = (e1_0 + e2_0) # * t1 # <exp(T1)HF|h1+h2|walker>/<HF|walker>
    e1 = (e1_2 + e2_2) # * t1 # <exp(T1)HF|T2 (h1+h2)|walker>/<HF|walker>
    # ecisd = h0 + (e1_0 + e2_0 + e1_2 + e2_2) / (1 + gt2g)
    return t2, e0, e1


# def ept2ccsd_energy_wrap(ham_data, weights, pt2_samples):
#     h0 = ham_data.h0
#     t1s, t2s, e0s, e1s = pt2_samples # shape = (nsamples, 4)
#     # tot_wt = jnp.sum(weights)
#     wt_t1 = weights * t1s
#     wt_t2 = weights * t2s
#     wt_e0 = weights * e0s
#     wt_e1 = weights * e1s
#     energy = h0 + jnp.sum(wt_e0) / jnp.sum(wt_t1) \
#                 + jnp.sum(wt_e1) / jnp.sum(wt_t1) \
#                 - jnp.sum(wt_t2) * jnp.sum(wt_e0) / jnp.sum(wt_t1)**2
#     return energy.real

In [ ]:
# def random_walker(walker):
#     # k1, k2 = jax.random.split(key)
#     # real and imaginary parts drawn from N(0,1)
#     re = np.random.rand(*walker.shape)
#     im = np.random.rand(*walker.shape)
#     w  = (re + 1j * im) / 2
#     return jnp.array(w)

In [ ]:
# walker = jnp.zeros((norb, nocc), dtype=complex)
# walker = walker.at[:nocc, :].set(jnp.eye(nocc))
# walker = random_walker(walker)
# t1, t2, e0, e1 = pt2_energy_kernel_rw_rh(walker, ham_data, meas_ctx, trial_data)
# ecisd = ham_data.h0 + (e0 + e1) / (t1 + t2)
# print(ecisd, (t1 + t2))

(-1.0960712830530561-1.8963505129868489e-10j) (0.9448370235469954-0.23336676527373382j)


In [132]:
from ad_afqmc_prototype.core.system import System
from ad_afqmc_prototype.core.ops import MeasOps
# from ad_afqmc_prototype.meas.rhf import force_bias_kernel_rw_rh, rdm1_kernel_rw
# from ad_afqmc_prototype.trial.rhf import overlap_r, get_rdm1

# k_force_bias = "force_bias"
k_energy = "energy"
# o_rdm1 = "rdm1"

def make_pt2ccsd_meas_ops(
    sys: System,
    memory_mode: str = "low",
    mixed_precision: bool = False,
    testing: bool = False,
) -> MeasOps:
    if sys.walker_kind.lower() != "restricted":
        raise ValueError(
            f"pt2CCSD MeasOps currently supports only restricted walkers, got: {sys.walker_kind}"
        )

    cfg = Pt2ccsdMeasCfg(
        memory_mode=memory_mode,
        mixed_real_dtype=jnp.float32 if mixed_precision else jnp.float64,
        mixed_complex_dtype=jnp.complex64 if mixed_precision else jnp.complex128,
        mixed_real_dtype_testing=jnp.float64 if testing else jnp.float32,
        mixed_complex_dtype_testing=jnp.complex128 if testing else jnp.complex64,
    )

    return MeasOps(
        overlap=overlap_r,
        build_meas_ctx=lambda ham_data, trial_data: build_meas_ctx(ham_data, trial_data, cfg),
        kernels={k_energy: energy_kernel_rw_rh},
        # observables={o_rdm1: rdm1_kernel_rw},
    )

In [133]:
from ad_afqmc_prototype.core.system import System
from ad_afqmc_prototype.trial.rhf import make_rhf_trial_ops
from ad_afqmc_prototype.meas.rhf  import make_rhf_meas_ops
# from ad_afqmc_prototype.meas.cisd  import make_cisd_meas_ops
from ad_afqmc_prototype.prop.afqmc import make_prop_ops, init_prop_state
from ad_afqmc_prototype.prop.types import QmcParams
# from ad_afqmc_prototype.prop.blocks import block as qmc_block

sys   = System(norb=norb, nelec=(nocc, nocc), walker_kind="restricted")
guide_ops = make_rhf_trial_ops(sys)
guide_meas_ops  = make_rhf_meas_ops(sys)
guide_prop_ops  = make_prop_ops(ham_data.basis, sys.walker_kind)
trial_meas_ops  = make_pt2ccsd_meas_ops(sys, mixed_precision=False)

params = QmcParams(
    dt            = 0.005,   # imaginary-time step
    n_walkers     = 200,      # walker population
    n_prop_steps  = 50,      # propagation steps per block
    n_blocks      = 200,     # sampling blocks
    n_eql_blocks  = 100,      # equilibration blocks (discarded)
    seed          = 17,
)

print(f"dt={params.dt}  n_walkers={params.n_walkers}  n_prop_steps={params.n_prop_steps}")
print(f"equlibrium imaginary time: {params.n_eql_blocks * params.n_prop_steps * params.dt:.2f} a.u.")
print(f"sampling imaginary time: {params.n_blocks * params.n_prop_steps * params.dt:.2f} a.u.")

dt=0.005  n_walkers=200  n_prop_steps=50
equlibrium imaginary time: 25.00 a.u.
sampling imaginary time: 50.00 a.u.


In [56]:
state = init_prop_state(
    sys        = sys,
    ham_data   = ham_data,
    trial_ops  = guide_ops,
    trial_data = guide_data,
    meas_ops   = guide_meas_ops,
    params     = params,
)

print(f"Initial walker batch shape: {state.walkers.shape}")
print(f"Initial weights (first 5):  {state.weights[:5]}")
print(f"Initial e_estimate: {float(state.e_estimate):.8f} Ha")
print(f"PYSCF-HF Energy: {float(mf.e_tot):.8f} Ha")
print(f"Initial mean |overlap|: {float(jnp.mean(jnp.abs(state.overlaps))):.6f}")

Initial walker batch shape: (200, 4, 2)
Initial weights (first 5):  [1. 1. 1. 1. 1.]
Initial e_estimate: -2.11285976 Ha
PYSCF-HF Energy: -2.11285976 Ha
Initial mean |overlap|: 1.000000


In [ ]:
# def init_prop_state_mixed(
#     *,
#     sys: System,
#     ham_data: HamChol,
#     guide_ops: TrialOps,
#     guide_data: Any,
#     guide_meas_ops: MeasOps,
#     params: QmcParamsBase,
#     initial_walkers: Any | None = None,
#     initial_e_estimate: jax.Array | None = None,
#     rdm1: jax.Array | None = None,
#     mesh: Mesh | None = None,
# ) -> PropState:
#     """
#     Initialize AFQMC propagation state.
#     """
#     n_walkers = params.n_walkers
#     seed = params.seed
#     key = jax.random.PRNGKey(int(seed))
#     weights = jnp.ones((n_walkers,))

#     if initial_walkers is None:
#         if rdm1 is None:
#             rdm1 = guide_ops.get_rdm1(guide_data)
#         initial_walkers = init_walkers(sys=sys, rdm1=rdm1, n_walkers=n_walkers)

#     overlaps = wk.vmap_chunked(guide_meas_ops.overlap, n_chunks=params.n_chunks, in_axes=(0, None))(
#         initial_walkers, guide_data
#     )

#     guide_e_est = None
#     if initial_e_estimate is not None:
#         guide_e_est = jnp.asarray(initial_e_estimate)
#     else:
#         guide_meas_ctx = guide_meas_ops.build_meas_ctx(ham_data, guide_data)
#         guide_e_kernel = guide_meas_ops.require_kernel(k_energy)
#         walker_0 = wk.take_walkers(initial_walkers, jnp.array([0]))
#         guide_e_samples = jnp.real(
#             wk.vmap_chunked(guide_e_kernel, n_chunks=1, in_axes=(0, None, None, None))(
#                 walker_0, ham_data, guide_meas_ctx, guide_data
#             )
#         )
#         e_est = jnp.mean(guide_e_samples)

#     pop_shift = e_est

#     node_encounters = jnp.asarray(0)

#     state = PropState(
#         walkers=initial_walkers,
#         weights=weights,
#         overlaps=overlaps,
#         rng_key=key,
#         pop_control_ene_shift=pop_shift,
#         e_estimate=e_est,
#         node_encounters=node_encounters,
#     )
#     return shard_prop_state(state, mesh)

In [ ]:
# from ad_afqmc_prototype.core.system import System
# from ad_afqmc_prototype.core.ops import TrialOps
# from ad_afqmc_prototype.trial.rhf import overlap_r, get_rdm1

# def make_pt2ccsd_trial_ops(sys: System) -> TrialOps:
#     if sys.nup != sys.ndn:
#         raise ValueError("Restricted pt2CCSD trial requires nup == ndn.")
#     if sys.walker_kind.lower() != "restricted":
#         raise ValueError(
#             f"pt2CCSD trial currently supports only restricted walkers, got: {sys.walker_kind}"
#         )
#     return TrialOps(overlap=overlap_r, get_rdm1=get_rdm1)

In [ ]:
# from ad_afqmc_prototype.core.ops import MeasOps
# from ad_afqmc_prototype.meas.rhf import force_bias_kernel_rw_rh, rdm1_kernel_rw
# from ad_afqmc_prototype.trial.rhf import overlap_r, get_rdm1

# k_force_bias = "force_bias"
# k_energy = "energy"
# o_rdm1 = "rdm1"

# def make_pt2ccsd_meas_ops(
#     sys: System,
#     memory_mode: str = "low",
#     mixed_precision: bool = False,
#     testing: bool = False,
# ) -> MeasOps:
#     if sys.walker_kind.lower() != "restricted":
#         raise ValueError(
#             f"pt2CCSD MeasOps currently supports only restricted walkers, got: {sys.walker_kind}"
#         )

#     cfg = Pt2ccsdMeasCfg(
#         memory_mode=memory_mode,
#         mixed_real_dtype=jnp.float32 if mixed_precision else jnp.float64,
#         mixed_complex_dtype=jnp.complex64 if mixed_precision else jnp.complex128,
#         mixed_real_dtype_testing=jnp.float64 if testing else jnp.float32,
#         mixed_complex_dtype_testing=jnp.complex128 if testing else jnp.complex64,
#     )

#     return MeasOps(
#         overlap=overlap_r,
#         build_meas_ctx=lambda ham_data, trial_data: build_meas_ctx(ham_data, trial_data, cfg),
#         kernels={k_force_bias: force_bias_kernel_rw_rh, k_energy: pt2_energy_kernel_rw_rh},
#         observables={o_rdm1: rdm1_kernel_rw},
#     )

In [ ]:
# trial_ops = make_pt2ccsd_trial_ops(sys)
# meas_ops  = make_pt2ccsd_meas_ops(sys, memory_mode="low")
# from ad_afqmc_prototype import walkers as wk

# e_kernel = meas_ops.require_kernel(k_energy)
# t1_sp, t2_sp, e0_sp, e1_sp = wk.vmap_chunked(e_kernel, n_chunks=params.n_chunks, in_axes=(0, None, None, None))(
#     state.walkers, ham_data, meas_ctx, trial_data
# )

In [117]:
from typing import Any, Callable, NamedTuple, Protocol

import jax
import jax.numpy as jnp
from jax import lax, tree_util

from ad_afqmc_prototype import walkers as wk
from ad_afqmc_prototype.core.levels import LevelPack
from ad_afqmc_prototype.core.ops import MeasOps, TrialOps, k_energy
from ad_afqmc_prototype.core.system import System
from ad_afqmc_prototype.walkers import SrFn
from ad_afqmc_prototype.prop.types import PropOps, PropState, QmcParams, QmcParamsFp
from ad_afqmc_prototype.prop.blocks import BlockObs

# def block_mixed(
#     state: PropState,
#     *,
#     sys: System,
#     params: QmcParams,
#     ham_data: Any,
#     guide_data: Any,
#     guide_ops: TrialOps,
#     guide_meas_ops: MeasOps,
#     guide_meas_ctx: Any,
#     guide_prop_ops: PropOps,
#     guide_prop_ctx: Any,
#     trial_data: Any,
#     trial_meas_ops: MeasOps,
#     trial_meas_ctx: Any,
#     observable_names: tuple[str, ...] = (),
#     sr_fn: Callable = wk.stochastic_reconfiguration,
# ) -> tuple[PropState, BlockObs]:
#     """
#     Block function for mixed sampling -- Trial =! Guide
#     propagation(Guide) + measurement(Trial)
#     """

#     # propagation is guided with the guiding wavefunction 
#     step = lambda st: guide_prop_ops.step(
#         st,
#         params     = params,
#         ham_data   = ham_data,
#         trial_data = guide_data,
#         trial_ops  = guide_ops,
#         meas_ops   = guide_meas_ops,
#         prop_ctx   = guide_prop_ctx,
#         meas_ctx   = guide_meas_ctx,
#     )

#     def _scan_step(carry: PropState, _x: Any):
#         carry = step(carry)
#         return carry, None

    # state, _ = lax.scan(_scan_step, state, xs=None, length=params.n_prop_steps)

    # walkers_new = wk.orthonormalize(state.walkers, sys.walker_kind)
    # guide_overlaps = wk.vmap_chunked(guide_meas_ops.overlap, n_chunks=params.n_chunks, in_axes=(0, None))(
    #     walkers_new, guide_data
    # )
    # state = state._replace(walkers=walkers_new, overlaps=guide_overlaps)

    # # some measurements with the guiding wavefunction if necessary
    # guide_e_kernel = guide_meas_ops.require_kernel(k_energy)
    # guide_e_samples = wk.vmap_chunked(guide_e_kernel, n_chunks=params.n_chunks, in_axes=(0, None, None, None))(
    #     state.walkers, ham_data, guide_meas_ctx, guide_data
    # ) # local energy with respect to the guiding wavefunction = <guide|H|walker>/<guide|walker>
    # guide_e_samples = jnp.real(guide_e_samples)

    # thresh = jnp.sqrt(2.0 / jnp.asarray(params.dt))
    # e_ref = state.e_estimate
    # is_nan = ~jnp.isfinite(guide_e_samples)
    # guide_e_samples = jnp.where(is_nan | (jnp.abs(guide_e_samples - e_ref) > thresh), e_ref, guide_e_samples)

    # guide_weights = jnp.where(is_nan, 0.0, state.weights)
    # guide_w_sum = jnp.sum(guide_weights)
    # guide_w_sum_safe = jnp.where(guide_w_sum == 0, 1.0, guide_w_sum)
    # guide_e_block = jnp.sum(guide_weights * guide_e_samples) / guide_w_sum_safe
    # guide_e_block = jnp.where(guide_w_sum == 0, e_ref, guide_e_block)

    # alpha = jnp.asarray(params.shift_ema, dtype=jnp.result_type(guide_e_block))
    # state = state._replace(
    #     weights=guide_weights,
    #     e_estimate=(1.0 - alpha) * state.e_estimate + alpha * guide_e_block,
    # )

    # # measuing with respect to trial
    # trial_e_kernel = trial_meas_ops.require_kernel(k_energy)
    # trial_e_samples = wk.vmap_chunked(trial_e_kernel, n_chunks=params.n_chunks, in_axes=(0, None, None, None))(
    #     state.walkers, ham_data, trial_meas_ctx, trial_data
    # )
    # trial_overlaps = wk.vmap_chunked(trial_meas_ops.overlap, n_chunks=params.n_chunks, in_axes=(0, None))(
    #     walkers_new, trial_data
    # )
    # trial_weights = guide_weights * trial_overlaps / guide_overlaps # w_trial = w_guide * <G|walker>/<T|walker>
    # trial_w_sum = jnp.sum(trial_weights)
    # trial_e_block = jnp.sum(trial_weights * trial_e_samples) / trial_w_sum
    # # trial_e_components = jnp.sum(trial_weights[:, None] * trial_e_samples, axis=0) / trial_w_sum
    # # t2, e0, e1 = 

    # obs_samples: dict[str, jax.Array] = {}
    # # for name in observable_names:
    # #     kernel = meas_ops.require_observable(name)
    # #     samples = wk.vmap_chunked(kernel, n_chunks=params.n_chunks, in_axes=(0, None, None, None))(
    # #         state.walkers, ham_data, guide_meas_ctx, guide_data
    # #     )
    # #     w_shape = (weights.shape[0],) + (1,) * max(samples.ndim - 1, 0)
    # #     num = jnp.sum(weights.reshape(w_shape) * samples, axis=0)
    # #     zero = jnp.zeros_like(num)
    # #     obs_samples[name] = jnp.where(wg_sum == 0, zero, num / w_sum_safe)
    
    # # performing SR at the end of Block propagation and measurement (Guide)
    # key, subkey = jax.random.split(state.rng_key)
    # zeta = jax.random.uniform(subkey)
    # w_sr, weights_sr = sr_fn(state.walkers, state.weights, zeta, sys.walker_kind)
    # overlaps_sr = wk.vmap_chunked(guide_meas_ops.overlap, n_chunks=params.n_chunks, in_axes=(0, None))(
    #     w_sr, guide_data
    # )
    # state = state._replace(
    #     walkers=w_sr,
    #     weights=weights_sr,
    #     overlaps=overlaps_sr,
    #     rng_key=key,
    # )

    # obs = BlockObs(
    #     scalars = {
    #         "guide_weight": guide_w_sum,
    #         "guide_energy": guide_e_block,
    #         "trial_weight": trial_w_sum,
    #         "trial_energy": trial_e_block,
    #         },
    #     observables = obs_samples,
    # )
    # return state, obs

In [164]:
def block_mixed(
    state: PropState,
    *,
    sys: System,
    params: QmcParams,
    ham_data: Any,
    guide_data: Any,
    guide_ops: TrialOps,
    guide_meas_ops: MeasOps,
    guide_meas_ctx: Any,
    guide_prop_ops: PropOps,
    guide_prop_ctx: Any,
    trial_data: Any,
    trial_meas_ops: MeasOps,
    trial_meas_ctx: Any,
    observable_names: tuple[str, ...] = (),
    sr_fn: Callable = wk.stochastic_reconfiguration,
) -> tuple[PropState, BlockObs]:
    """
    Block function for mixed sampling -- Trial =! Guide
    propagation(Guide) + measurement(Trial)
    """

    # propagation is guided with the guiding wavefunction 
    step = lambda st: guide_prop_ops.step(
        st,
        params     = params,
        ham_data   = ham_data,
        trial_data = guide_data,
        trial_ops  = guide_ops,
        meas_ops   = guide_meas_ops,
        prop_ctx   = guide_prop_ctx,
        meas_ctx   = guide_meas_ctx,
    )

    def _scan_step(carry: PropState, _x: Any):
        carry = step(carry)
        return carry, None

    state, _ = lax.scan(_scan_step, state, xs=None, length=params.n_prop_steps)

    walkers_new = wk.orthonormalize(state.walkers, sys.walker_kind)
    guide_overlaps = wk.vmap_chunked(guide_meas_ops.overlap, n_chunks=params.n_chunks, in_axes=(0, None))(
        walkers_new, guide_data
    )
    state = state._replace(walkers=walkers_new, overlaps=guide_overlaps)

    # some measurements with the guiding wavefunction if necessary
    guide_e_kernel = guide_meas_ops.require_kernel(k_energy)
    guide_e_samples = wk.vmap_chunked(guide_e_kernel, n_chunks=params.n_chunks, in_axes=(0, None, None, None))(
        state.walkers, ham_data, guide_meas_ctx, guide_data
    ) # local energy with respect to the guiding wavefunction = <guide|H|walker>/<guide|walker>
    guide_e_samples = jnp.real(guide_e_samples)

    thresh = jnp.sqrt(2.0 / jnp.asarray(params.dt))
    e_ref = state.e_estimate
    is_nan = ~jnp.isfinite(guide_e_samples)
    guide_e_samples = jnp.where(is_nan | (jnp.abs(guide_e_samples - e_ref) > thresh), e_ref, guide_e_samples)

    guide_weights = jnp.where(is_nan, 0.0, state.weights)
    guide_w_block = jnp.sum(guide_weights)
    guide_w_block_safe = jnp.where(guide_w_block == 0, 1.0, guide_w_block)
    guide_e_block = jnp.sum(guide_weights * guide_e_samples) / guide_w_block_safe
    guide_e_block = jnp.where(guide_w_block == 0, e_ref, guide_e_block)

    alpha = jnp.asarray(params.shift_ema, dtype=jnp.result_type(guide_e_block))
    state = state._replace(
        weights=guide_weights,
        e_estimate=(1.0 - alpha) * state.e_estimate + alpha * guide_e_block,
    )

    # measuing with respect to trial
    trial_e_kernel = trial_meas_ops.require_kernel(k_energy)
    trial_t2s, trial_e0s, trial_e1s = wk.vmap_chunked(
        trial_e_kernel, n_chunks=params.n_chunks, in_axes=(0, None, None, None))(
        state.walkers, ham_data, trial_meas_ctx, trial_data
    )
    trial_overlaps = wk.vmap_chunked(trial_meas_ops.overlap, n_chunks=params.n_chunks, in_axes=(0, None))(
        walkers_new, trial_data
    )
    trial_weights = guide_weights * trial_overlaps / guide_overlaps # w_trial = w_guide * <G|walker>/<T|walker>
    trial_w_block = jnp.sum(trial_weights)
    trial_t2_block = jnp.sum(trial_weights * trial_t2s) / trial_w_block
    trial_e0_block = jnp.sum(trial_weights * trial_e0s) / trial_w_block
    trial_e1_block = jnp.sum(trial_weights * trial_e1s) / trial_w_block
    # trial_e_block = jnp.sum(trial_weights * trial_e_samples) / trial_w_sum
    # trial_e_parts_block = jnp.sum(trial_weights[:, None] * trial_e_parts, axis=0) / trial_w_block
    # trial_t2_block, trial_e0_block, trial_e1_block = trial_e_parts_block

    obs_samples: dict[str, jax.Array] = {}
    # for name in observable_names:
    #     kernel = meas_ops.require_observable(name)
    #     samples = wk.vmap_chunked(kernel, n_chunks=params.n_chunks, in_axes=(0, None, None, None))(
    #         state.walkers, ham_data, guide_meas_ctx, guide_data
    #     )
    #     w_shape = (weights.shape[0],) + (1,) * max(samples.ndim - 1, 0)
    #     num = jnp.sum(weights.reshape(w_shape) * samples, axis=0)
    #     zero = jnp.zeros_like(num)
    #     obs_samples[name] = jnp.where(wg_sum == 0, zero, num / w_sum_safe)
    
    # performing SR at the end of Block propagation and measurement (Guide)
    key, subkey = jax.random.split(state.rng_key)
    zeta = jax.random.uniform(subkey)
    w_sr, weights_sr = sr_fn(state.walkers, state.weights, zeta, sys.walker_kind)
    overlaps_sr = wk.vmap_chunked(guide_meas_ops.overlap, n_chunks=params.n_chunks, in_axes=(0, None))(
        w_sr, guide_data
    )
    state = state._replace(
        walkers=w_sr,
        weights=weights_sr,
        overlaps=overlaps_sr,
        rng_key=key,
    )

    obs = BlockObs(
        scalars = {
            "guide_weight": guide_w_block,
            "guide_energy": guide_e_block,
            "trial_weight": trial_w_block,
            "trial_t2"    : trial_t2_block,
            "trial_e0"    : trial_e0_block,
            "trial_e1"    : trial_e1_block,
            },
        observables = obs_samples,
    )
    return state, obs

In [165]:
from typing import Any

import jax
import jax.numpy as jnp
# from jax import lax, tree_util

# from ad_afqmc_prototype import walkers as wk
# from ad_afqmc_prototype.core.levels import LevelPack
from ad_afqmc_prototype.core.ops import MeasOps, TrialOps, k_energy
from ad_afqmc_prototype.core.system import System
# from ad_afqmc_prototype.walkers import SrFn
from ad_afqmc_prototype.prop.types import PropOps, QmcParamsBase
from ad_afqmc_prototype.prop.blocks import BlockFn
# from ad_afqmc_prototype.stat_utils import blocking_analysis_ratio, jackknife_ratios, rebin_observable, reject_outliers
# from ad_afqmc_prototype.walkers import stochastic_reconfiguration

# from jax import lax
# from jax.sharding import Mesh, NamedSharding
from jax.sharding import PartitionSpec as P

# from ad_afqmc_prototype.driver import QmcResult, make_run_blocks
from functools import partial

def make_run_mixed_blocks(
    *,
    mixed_block_fn: Any,
    sys: System,
    params: QmcParamsBase,
    guide_ops: TrialOps,
    guide_meas_ops: MeasOps,
    guide_prop_ops: PropOps,
    trial_meas_ops: MeasOps,
    observable_names: tuple[str, ...] = (),
) -> Callable:
    """
    Build a jitted run_blocks for mixed sampling (Trial =! Guide).
    We keep ham_data, trial_data, meas_ctx, prop_ctx as arguments to
    improve compilation, as these objects can be large.
    """

    @partial(jax.jit, static_argnames=("n_blocks",))
    def run_mixed_blocks(
        state0,
        *,
        ham_data,
        guide_data,
        guide_meas_ctx,
        guide_prop_ctx,
        trial_data,
        trial_meas_ctx,
        n_blocks: int,
    ):
        def one_block(state, _):
            state, obs = mixed_block_fn(
                state,
                sys              = sys,
                params           = params,
                ham_data         = ham_data,
                guide_data       = guide_data,
                guide_ops        = guide_ops,
                guide_meas_ops   = guide_meas_ops,
                guide_meas_ctx   = guide_meas_ctx,
                guide_prop_ops   = guide_prop_ops,
                guide_prop_ctx   = guide_prop_ctx,
                trial_data       = trial_data,
                trial_meas_ops   = trial_meas_ops,
                trial_meas_ctx   = trial_meas_ctx,
                observable_names = observable_names,
            )
            obs_tuple = tuple(obs.observables[name] for name in observable_names)
            return state, (obs.scalars, obs_tuple)

        stateN, (scalars, obs) = lax.scan(one_block, state0, xs=None, length=n_blocks)
        return stateN, scalars, obs

    return run_mixed_blocks

In [162]:
def get_init_pt2trial_energy(
        init_state, 
        ham_data, 
        trial_data, 
        trial_meas_ops, 
        trial_meas_ctx, 
        ):
    walker_0 = wk.take_walkers(init_state.walkers, jnp.array([0]))
    trial_e_kernel = trial_meas_ops.require_kernel(k_energy)
    t2, e0, e1 = wk.vmap_chunked(trial_e_kernel, n_chunks=1, in_axes=(0, None, None, None))(
            walker_0, ham_data, trial_meas_ctx, trial_data)
     
    trial_overlap = wk.vmap_chunked(trial_meas_ops.overlap, n_chunks=params.n_chunks, in_axes=(0, None))(
        walker_0, trial_data
    )
    guide_overlap = init_state.overlaps[0]
    trial_weights = init_state.weights * trial_overlap / guide_overlap
    trial_energy = (ham_data.h0 + e0 + e1 - t2 * e1).mean()

    return trial_energy+0j, jnp.sum(trial_weights)

In [ ]:
# walker_0 = wk.take_walkers(state.walkers, jnp.array([0]))
# trial_e_kernel = trial_meas_ops.require_kernel(k_energy)
# t2, e0, e1 = wk.vmap_chunked(trial_e_kernel, n_chunks=1, in_axes=(0, None, None, None))(
#         walker_0, ham_data, trial_meas_ctx, trial_data
#     )

In [ ]:
# ham_data.h0 + e0 + e1 - t2 * e1

Array([-2.19214257+0.j], dtype=complex128)

In [166]:
from typing import Any

import jax
import jax.numpy as jnp
# from jax import lax, tree_util

# from ad_afqmc_prototype import walkers as wk
# from ad_afqmc_prototype.core.levels import LevelPack
# from ad_afqmc_prototype.core.ops import MeasOps, TrialOps, k_energy
# from ad_afqmc_prototype.core.system import System
# from ad_afqmc_prototype.walkers import SrFn
# from ad_afqmc_prototype.prop.types import PropOps, PropState, QmcParams
# from ad_afqmc_prototype.prop.blocks import BlockFn
# from ad_afqmc_prototype.stat_utils import blocking_analysis_ratio, reject_outliers
from ad_afqmc_prototype.walkers import stochastic_reconfiguration

# from jax import lax
from jax.sharding import Mesh, NamedSharding
from jax.sharding import PartitionSpec as P

# from ad_afqmc_prototype.driver import make_run_blocks
from functools import partial
import time
print = partial(print, flush=True)

# def run_qmc(
    # *,
#     sys: System,
#     params: QmcParams,
#     ham_data: Any,
#     trial_data: Any,
#     meas_ops: MeasOps,
#     trial_ops: TrialOps,
#     prop_ops: PropOps,
#     block_fn: BlockFn,
#     state: PropState | None = None,
#     meas_ctx: Any | None = None,
#     target_error: float | None = None,
#     mesh: Mesh | None = None,
#     observable_names: tuple[str, ...] = (),
# ) -> QmcResult:
    # """
    # equilibration blocks then sampling blocks.

    # Returns:
    #   QmcResult with energy statistics plus block-level observable estimates.
    # """
    
mesh = None
observable_names = ()
target_error = None
block_fn = block_mixed
state = None
guide_meas_ctx = None
trial_meas_ctx = None

for name in observable_names:
    guide_meas_ops.require_observable(name)

# build ctx
guide_prop_ctx = guide_prop_ops.build_prop_ctx(ham_data, guide_ops.get_rdm1(guide_data), params)
if guide_meas_ctx is None:
    guide_meas_ctx = guide_meas_ops.build_meas_ctx(ham_data, guide_data)

if trial_meas_ctx is None:
    trial_meas_ctx = trial_meas_ops.build_meas_ctx(ham_data, trial_data)

if state is None:
    state = guide_prop_ops.init_prop_state(
        sys        = sys,
        ham_data   = ham_data,
        trial_ops  = guide_ops,
        trial_data = guide_data,
        meas_ops   = guide_meas_ops,
        params     = params,
        mesh       = mesh,
    )

trial_energy0, trial_weights0 = get_init_pt2trial_energy(
    init_state     = state, 
    ham_data       = ham_data, 
    trial_data     = trial_data,
    trial_meas_ops = trial_meas_ops, 
    trial_meas_ctx = trial_meas_ctx, 
    )

if mesh is None or mesh.size == 1:
    block_fn_sr = block_fn
else:
    data_sh = NamedSharding(mesh, P("data"))
    sr_sharded = partial(stochastic_reconfiguration, data_sharding=data_sh)
    block_fn_sr = partial(block_fn, sr_fn=sr_sharded)

run_blocks = make_run_mixed_blocks(
    mixed_block_fn   = block_fn_sr,
    sys              = sys,
    params           = params,
    guide_ops        = guide_ops,
    guide_meas_ops   = guide_meas_ops,
    guide_prop_ops   = guide_prop_ops,
    trial_meas_ops   = trial_meas_ops,
    observable_names = observable_names,
)

t0 = time.perf_counter()
t_mark = t0

block_time = params.dt * params.n_prop_steps
print_every = params.n_eql_blocks // 5 if params.n_eql_blocks >= 5 else 0
guide_block_e_eq = []
guide_block_w_eq = []
trial_block_w_eq = []
trial_block_t2_eq = []
trial_block_e0_eq = []
trial_block_e1_eq = []
# guide_block_obs_eq = {name: [] for name in observable_names}
guide_block_e_eq.append(state.e_estimate)
guide_block_w_eq.append(jnp.sum(state.weights))
# trial_block_e_eq.append(trial_energy0)
# trial_block_w_eq.append(jnp.sum(trial_weights0))
print("\nEquilibration:\n")
if print_every:
    print(
        f"{'':4s}"
        f"{'block':>9s}  "
        f"{'InverseTmp':>12s}  "
        f"{'Guide_E_blk':>14s}  "
        f"{'Guide_W_blk':>12s}   "
        f"{'Trial_E_blk':>14s}  "
        f"{'Trial_W_blk':>12s}   "
        f"{'nodes':>10s}  "
        f"{'t[s]':>8s}"
    )
print(
    f"[eql {0:4d}/{params.n_eql_blocks}]  "
    f"{0 * block_time:12.2f}  "
    f"{float(guide_block_e_eq[0].real):14.10f}  "
    f"{float(guide_block_w_eq[0].real):12.6e}  "
    f"{float(trial_energy0.real):14.10f}  "
    f"{float(trial_weights0.real):12.6e}  "
    f"{int(state.node_encounters):10d}  "
    f"{0.0:8.1f}"
)
chunk = print_every if print_every > 0 else 1
for start in range(0, params.n_eql_blocks, chunk):
    n = min(chunk, params.n_eql_blocks - start)
    state, scalars_chunk, obs_chunk = run_blocks(
        state,
        ham_data       = ham_data,
        guide_data     = guide_data,
        guide_meas_ctx = guide_meas_ctx,
        guide_prop_ctx = guide_prop_ctx,
        trial_data     = trial_data,
        trial_meas_ctx = trial_meas_ctx,
        n_blocks       = n,
    )
    # guide
    guide_block_e_eq.extend(scalars_chunk["guide_energy"].tolist())
    guide_block_w_eq.extend(scalars_chunk["guide_weight"].tolist())
    guide_e_chunk = scalars_chunk["guide_energy"]
    guide_w_chunk = scalars_chunk["guide_weight"]
    guide_w_chunk_avg = jnp.mean(guide_w_chunk)
    guide_e_chunk_avg = jnp.mean(guide_e_chunk * guide_w_chunk) / guide_w_chunk_avg
    # trial
    trial_block_w_eq.extend(scalars_chunk["trial_weight"].tolist())
    trial_block_t2_eq.extend(scalars_chunk["trial_t2"].tolist())
    trial_block_e0_eq.extend(scalars_chunk["trial_e0"].tolist())
    trial_block_e1_eq.extend(scalars_chunk["trial_e1"].tolist())
    # for i, name in enumerate(observable_names):
    #     block_obs_eq[name].append(obs_chunk[i])
    trial_w_chunk = scalars_chunk["trial_weight"]
    trial_t2_chunk = scalars_chunk["trial_t2"]
    trial_e0_chunk = scalars_chunk["trial_e0"]
    trial_e1_chunk = scalars_chunk["trial_e1"]
    trial_w_chunk_avg = jnp.mean(trial_w_chunk)
    trial_t2_chunk_avg = jnp.mean(trial_w_chunk * trial_t2_chunk) / trial_w_chunk_avg
    trial_e0_chunk_avg = jnp.mean(trial_w_chunk * trial_e0_chunk) / trial_w_chunk_avg
    trial_e1_chunk_avg = jnp.mean(trial_w_chunk * trial_e1_chunk) / trial_w_chunk_avg
    pt2trial_energy_avg = ham_data.h0 + trial_e0_chunk_avg + trial_e1_chunk_avg - trial_t2_chunk_avg * trial_e0_chunk_avg
    elapsed = time.perf_counter() - t0
    print(
        f"[eql {start + n:4d}/{params.n_eql_blocks}]  "
        f"{(start + n) * block_time:12.2f}  "
        f"{float(guide_e_chunk_avg):14.10f}  "
        f"{float(guide_w_chunk_avg):12.6e}  "
        f"{float(pt2trial_energy_avg.real):14.10f}  "
        f"{float(trial_w_chunk_avg.real):12.6e}  "
        f"{int(state.node_encounters):10d}  "
        f"{elapsed:8.1f}"
    )

guide_block_e_eq = jnp.asarray(guide_block_e_eq)
guide_block_w_eq = jnp.asarray(guide_block_w_eq)

# block_obs_eq = {
#     name: (jnp.concatenate(block_obs_eq[name], axis=0) if len(block_obs_eq[name]) > 0 else None)
#     for name in observable_names
# }

# sampling
# print("\nSampling:\n")
# if target_error is None:
#     target_error = 0.0
# print_every = params.n_blocks // 10 if params.n_blocks >= 10 else 0
# block_eg_s = []
# block_wg_s = []
# block_obs_s = {name: [] for name in observable_names}
# if print_every:
#     print(
#         f"{'':4s}{'block':>9s}  {'E_avg':>14s}  {'E_err':>10s}  {'E_block':>14s}  "
#         f"{'W':>12s}    {'nodes':>10s}  {'dt[s/bl]':>10s}  {'t[s]':>7s}"
#     )

# chunk = print_every if print_every > 0 else 1
# for start in range(0, params.n_blocks, chunk):
#     n = min(chunk, params.n_blocks - start)
#     state, scalars_chunk, obs_chunk = run_blocks(
#         state,
#         ham_data=ham_data,
#         trial_data=trial_data,
#         meas_ctx=meas_ctx,
#         prop_ctx=prop_ctx,
#         n_blocks=n,
#     )
#     e_chunk = scalars_chunk["energy"]
#     w_chunk = scalars_chunk["weight"]
#     block_e_s.extend(e_chunk.tolist())
#     block_w_s.extend(w_chunk.tolist())
#     for i, name in enumerate(observable_names):
#         block_obs_s[name].append(obs_chunk[i])
#     w_chunk_avg = jnp.mean(w_chunk)
#     e_chunk_avg = jnp.mean(e_chunk * w_chunk) / w_chunk_avg
#     elapsed = time.perf_counter() - t0
#     dt_per_block = (time.perf_counter() - t_mark) / float(n)
#     t_mark = time.perf_counter()
#     stats = blocking_analysis_ratio(
#         jnp.asarray(block_e_s), jnp.asarray(block_w_s), print_q=False
#     )
#     mu = stats["mu"]
#     se = stats["se_star"]
#     nodes = int(state.node_encounters)
#     print(
#         f"[blk {start + n:4d}/{params.n_blocks}]  "
#         f"{mu:14.10f}  "
#         f"{(f'{se:10.3e}' if se is not None else ' ' * 10)}  "
#         f"{float(e_chunk_avg):16.10f}  "
#         f"{float(w_chunk_avg):12.6e}  "
#         f"{nodes:10d}  "
#         f"{dt_per_block:9.3f}  "
#         f"{elapsed:8.1f}"
#     )
#     if se is not None and se <= target_error and target_error > 0.0:
#         print(f"\nTarget error {target_error:.3e} reached at block {start + n}.")
#         break
# block_e_s = jnp.asarray(block_e_s)
# block_w_s = jnp.asarray(block_w_s)
# block_obs_s = {
#     name: (jnp.concatenate(block_obs_s[name], axis=0) if len(block_obs_s[name]) > 0 else None)
#     for name in observable_names
# }

# data_clean, keep_mask = reject_outliers(jnp.column_stack((block_e_s, block_w_s)), obs=0)
# print(f"\nRejected {block_e_s.shape[0] - data_clean.shape[0]} outlier blocks.")
# block_e_s = jnp.asarray(data_clean[:, 0])
# block_w_s = jnp.asarray(data_clean[:, 1])
# keep_mask = jnp.asarray(keep_mask)
# block_obs_s = {
#     name: (arr[keep_mask] if arr is not None else None) for name, arr in block_obs_s.items()
# }
# print("\nFinal blocking analysis:")
# stats = blocking_analysis_ratio(block_e_s, block_w_s, print_q=True)
# mean, err = stats["mu"], stats["se_star"]


Equilibration:

        block    InverseTmp     Guide_E_blk   Guide_W_blk      Trial_E_blk   Trial_W_blk        nodes      t[s]
[eql    0/100]          0.00   -2.1128597642  2.000000e+02   -2.1921425660  2.000000e+02           0       0.0
[eql   20/100]          5.00   -2.1780159187  2.002969e+02   -2.1930235305  2.002969e+02           0       2.2
[eql   40/100]         10.00   -2.1907252297  2.000166e+02   -2.1916938012  2.000166e+02           0       4.3
[eql   60/100]         15.00   -2.1828101947  1.999514e+02   -2.1922830222  1.999514e+02           0       5.0
[eql   80/100]         20.00   -2.1815332973  1.999700e+02   -2.1925087691  1.999700e+02           0       5.8
[eql  100/100]         25.00   -2.1794072651  1.999827e+02   -2.1924280886  1.999827e+02           0       6.5


In [111]:
scalars_chunk['trial_energy'].shape

(20, 200)

In [102]:
chunk

20

In [100]:
min(chunk, params.n_eql_blocks - start)

20

In [97]:
state, scalars_chunk, obs_chunk = run_blocks(
        state,
        ham_data       = ham_data,
        guide_data     = guide_data,
        guide_meas_ctx = guide_meas_ctx,
        guide_prop_ctx = guide_prop_ctx,
        trial_data     = trial_data,
        trial_meas_ctx = trial_meas_ctx,
        n_blocks       = n,
    )

In [99]:
scalars_chunk

{'guide_energy': Array([-2.199986  , -2.20239781, -2.1756847 , -2.17449634, -2.17401893,
        -2.17617224, -2.18053186, -2.17667528, -2.18094891, -2.1736127 ,
        -2.18058009, -2.18866076, -2.19051629, -2.17917997, -2.18897981,
        -2.1880632 , -2.18567899, -2.1756493 , -2.17762188, -2.18974719],      dtype=float64),
 'guide_weight': Array([200.11688118, 200.18483651, 199.80460117, 199.80864469,
        199.90973106, 199.99286879, 200.09141111, 199.78227242,
        199.90406472, 199.82032197, 200.03982161, 199.97912866,
        200.09604074, 200.08668568, 199.98144434, 200.10957717,
        199.96111482, 199.78120098, 199.92844419, 200.08844784],      dtype=float64),
 'trial_energy': Array([-2.19067658-1.45423661e-05j, -2.1906967 -2.37595272e-04j,
        -2.19148973-1.13502342e-04j, -2.19136456-1.66574853e-05j,
        -2.1913939 -1.20604891e-04j, -2.19155635-7.72646527e-05j,
        -2.19210402-2.75238827e-04j, -2.19223225+1.49660831e-04j,
        -2.19369373+6.07371668e-